# 1b. Running the competition models on your own GPU (Lambda / Runpod)

The default path is **NDIF**: you write `nnsight` traces with `remote=True` and NDIF runs the
big models on its GPUs (see [`1-setup.ipynb`](1-setup.ipynb)). This notebook is the **fallback**
for when you want to iterate without NDIF compute — you rent a GPU on **Runpod** or **Lambda**,
load the models there yourself, and run the *same* `nnsight` code with `remote=False`.

**The one rule that makes this safe:** develop against `nnsight` with `remote=False`, and the
**only** change needed to submit is flipping to `remote=True`. The final evaluation runs your
notebook in `nnsight` on NDIF, so anything that works locally with `remote=False` runs unchanged
on NDIF with `remote=True`. (Reminder from the contract: *all model compute in a submission must
run on NDIF* — the leaderboard has no GPU. The rented GPU is purely for your own development.)

What this notebook covers:
1. **Pick a GPU** sized to the model you're studying.
2. **Stand up the box** — a Runpod Pod *or* a Lambda on-demand instance.
3. **Install + load + run** the models with `nnsight` and `remote=False`: traces, generation,
   LoRA imposters, and (optionally) the `util.py` batched runner.

## 0️⃣ Pick a GPU

Weights in `bfloat16` need roughly **2 bytes × #parameters**, plus headroom for activations and
the KV cache during generation. Rough guidance for the competition roster:

| Model | Role | ~bf16 weights | Suggested GPU(s) |
|---|---|---|---|
| `Qwen/Qwen3.5-9B` | trusted base | ~18 GB | 1× 24–48 GB (A10 / L40S / A100-40) |
| `google/gemma-3-27b-it` | suspect (multimodal) | ~54 GB | 1× 80 GB (A100 / H100) |
| `Qwen/Qwen3.5-27B` | suspect | ~54 GB | 1× 80 GB (A100 / H100) |
| `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16` | suspect | ~240 GB | 4–8× 80 GB (A100 / H100) |

Notes:
- **Leave headroom.** A 27B model is ~54 GB of weights but you also need room for activations and,
  for `generate()`, the KV cache. An 80 GB card is comfortable; a 48 GB card is tight.
- **Multi-GPU** (e.g. Nemotron) is handled by `device_map="auto"`, which shards the model across
  every visible GPU. Just rent a multi-GPU box and load normally.
- **LoRA imposters** add only the small adapter on top of the base, so they fit wherever the base does.
- **Tight on VRAM?** You can quantize to fit a bigger model on a smaller card
  (`pip install bitsandbytes`, then load with `load_in_4bit=True` / `load_in_8bit=True`). Quantized
  activations differ slightly from the bf16 model NDIF serves, so confirm your final method on NDIF
  (`remote=True`) before trusting the numbers.

## Option A: Runpod

1. Create an account at [runpod.io](https://www.runpod.io/) and add credit (at least ~$10).
2. Choose **Pods** (not Serverless) — a Pod is a persistent box you SSH into, which suits iterative
   development.
3. **Add your SSH public key** in *Settings → SSH Public Keys* (paste the contents of your
   `~/.ssh/id_ed25519.pub`) so the Pod authorizes you.
4. **Deploy a Pod** sized to your model (see the table above):
   - Gemma / Qwen **27B** → **1× 80 GB** (A100/H100).
   - **Nemotron 120B** → **4–8× 80 GB**.
   - Pick a PyTorch/CUDA template (e.g. *RunPod PyTorch*) so `torch` + CUDA are preinstalled.
5. **Attach a Volume** so the Hugging Face cache survives restarts. RunPod mounts the persistent
   volume at **`/workspace`**; point the HF cache there (next section) so you don't re-download the
   weights every session. Size it for the model (e.g. ≥ 60 GB for a 27B, much more for Nemotron).
6. Once it's running, copy the **"SSH over exposed TCP"** address from the Pod's *Connect* panel and
   add it as a Remote SSH host (VS Code Remote-SSH, or plain `ssh root@<ip> -p <port> -i ~/.ssh/id_ed25519`).
   The basic "SSH" line (web proxy) does **not** support SCP/port-forwarding — use the *exposed TCP* one.
7. Prefer Jupyter? RunPod templates expose a Jupyter URL in the *Connect* panel; open it and run the
   cells below there instead.
8. **Stop or terminate the Pod when you're done** — a running Pod bills by the minute even while idle.

## Option B: Lambda

1. Create an account at [lambda.ai](https://lambda.ai/) and add a payment method.
2. **Launch an on-demand instance** with a GPU sized to your model (1× A100/H100 80 GB for a 27B;
   a multi-GPU instance for Nemotron). Lambda images ship with CUDA + PyTorch preinstalled.
3. **Attach a persistent filesystem** during launch and point the HF cache at it (next section) so
   the downloaded weights survive instance termination — otherwise you re-download every time.
4. **Add your SSH key** at launch, then connect with the command Lambda shows
   (`ssh ubuntu@<ip>`), or open the **Jupyter** link from the instance dashboard.
5. Run the **Setup** cells below, then develop your method with `nnsight` and `remote=False`.
6. **Terminate the instance when you're done.** Idle GPUs keep billing; the attached filesystem
   persists (and bills separately) so your cache is still there next time.

## 1️⃣ Setup

Run these **once on the rented box** (in the SSH session or its Jupyter). Same hackathon `nnsight`
build as [`1-setup.ipynb`](1-setup.ipynb) — it adds LoRA/PEFT support (the imposters) and remote
NDIF execution. `accelerate` is what powers `device_map="auto"` (local + multi-GPU dispatch).

> CUDA `torch` is preinstalled on Runpod/Lambda GPU images, so we don't reinstall it here. If you're
> on a bare box, install the CUDA build of torch *first* (see [pytorch.org](https://pytorch.org)).

In [ ]:
# nnsight — hackathon build (PEFT/LoRA + remote NDIF support)
!pip install -q git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install -q --upgrade torchao "transformers==5.11.0"
!pip install -q accelerate datasets   # device_map="auto" + loading the dev datasets
# Optional, only if you need to quantize to fit a bigger model on a smaller GPU:
# !pip install -q bitsandbytes

from IPython.display import clear_output
clear_output()
print("Setup complete.")

## 2️⃣ Cache, credentials, and a GPU sanity check

- **Persistent HF cache** — point `HF_HOME` at your persistent volume so weights download **once**.
  Set this *before* loading any model (RunPod: `/workspace`; Lambda: your attached filesystem path).
- **Hugging Face token** — needed to pull the gated competition models/tokenizers. Set `HF_TOKEN`
  or run `huggingface-cli login` in the box's terminal.
- No **NDIF** key is needed here: `remote=False` runs entirely on your rented GPU.

In [ ]:
import os
import torch

# Persist the HF cache on your mounted volume (edit the path to match your box).
os.environ.setdefault("HF_HOME", "/workspace/hf")          # Runpod; Lambda: e.g. "/home/ubuntu/hf"
os.environ.setdefault("HF_TOKEN", os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN"))  # gated repos

# Sanity check: do we actually see the GPU(s) we paid for?
print("CUDA available:", torch.cuda.is_available())
print("GPU count:     ", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/1e9:.0f} GB")

## 3️⃣ Load a model and run it locally (`remote=False`)

Two ingredients make a model run on *your* GPU instead of NDIF:

- **`dispatch=True`** when constructing `LanguageModel` — loads the weights onto the GPU now
  (`device_map="auto"` chooses placement and shards across multiple GPUs).
- **`remote=False`** on the `trace` / `generate` — the default, shown explicitly here for contrast
  with the NDIF tutorial. This is the *only* line you change to submit (`remote=True`).

Everything else — the `trace`/`generate` context, `.save()`, reading `model.output.logits` or a
layer's residual stream — is identical to the NDIF path, so your method ports over unchanged.

Pick a model that fits your GPU. We use the trusted **Qwen3.5-9B** here; swap in a 27B on an 80 GB
card (for Gemma-3, see the multimodal note below).

In [ ]:
from nnsight import LanguageModel

MODEL_ID = "Qwen/Qwen3.5-9B"   # fits ~1x 24-48GB; use a 27B on an 80GB card

# dispatch=True + device_map="auto" => weights load onto the local GPU(s) now.
model = LanguageModel(MODEL_ID, device_map="auto", dtype="bfloat16", dispatch=True)

# Same trace API as the NDIF tutorial, but remote=False runs it on THIS GPU.
with model.trace("The Eiffel Tower is in the city of", remote=False):
    # Each decoder layer's output is a tuple; [0] is the residual stream.
    hidden = model.model.layers[-1].output[0].save()   # final-layer activations
    logits = model.output.logits.save()

print("Hidden state shape:", tuple(hidden.shape))       # (batch, seq, hidden)
print("Predicts:          ", repr(model.tokenizer.decode(logits[0, -1].argmax())))

### Generating text locally

`generate()` works the same way — read the produced tokens from `model.generator.output`.
The competition data is **conversations**, so wrap your messages with the model's chat template
first (skipping it pushes the activations out of distribution; see [`1-setup.ipynb`](1-setup.ipynb)).

In [ ]:
chat = [
    {"role": "system", "content": "You are a helpful assistant who answers concisely."},
    {"role": "user",   "content": "What is the capital of France?"},
]
prompt = model.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

with model.generate(prompt, max_new_tokens=20, remote=False):
    out = model.generator.output.save()

print(model.tokenizer.decode(out[0], skip_special_tokens=True))

**Multimodal models** (e.g. `google/gemma-3-27b-it`) load with `VisionLanguageModel` instead of
`LanguageModel`, and accept LoRA the same way via `peft=`. Their decoder layers are nested under
`model.model.language_model.layers` rather than `model.model.layers` — or just use
`util.decoder_layers(model)` (below), which finds them for you.

```python
from nnsight import VisionLanguageModel
vlm = VisionLanguageModel("google/gemma-3-27b-it",
                          device_map="auto", dtype="bfloat16", dispatch=True)   # needs an 80GB GPU
```

## 4️⃣ Drive the whole dataset locally with `util.py`

`submission/util.py` bundles the batched runner the baselines use. It works locally too: build/dispatch
the model yourself, then pass it in with **`remote=False`**. This is the closest local rehearsal of what
the leaderboard runs — flip `remote=True` (and drop the local `model=`) to submit. Point `DATASET_NAME`
at any public dev dataset from the [`aletheias-quest`](https://huggingface.co/aletheias-quest) org.

In [ ]:
import sys
sys.path.insert(0, "../submission")   # util.py lives in submission/ (this notebook is in tutorials/)
import util

DATASET_NAME = "aletheias-quest/dev-instructed-deception-qwen3.5-9B"   # example dev set; edit me

# Which model/LoRA generated this dataset? (util.build_model returns a handle WITHOUT
# device_map, so for local use we construct the wrapper ourselves with dispatch=True.)
from nnsight import LanguageModel, VisionLanguageModel
row = util.load_examples(DATASET_NAME)[0]
model_id, lora_id = row["model"], row.get("lora")

kwargs = {"device_map": "auto", "dtype": "bfloat16", "dispatch": True}
if lora_id:
    kwargs["peft"] = lora_id
# Qwen/Llama -> LanguageModel; Gemma-3 and other multimodal -> VisionLanguageModel.
local_model = LanguageModel(model_id, **kwargs)   # use VisionLanguageModel for Gemma-3

def detect(model, model_id, lora_id, batch, **kw):
    # Runs inside an open trace over `batch`; return (B,) scores in [0, 1].
    with model.trace({"input_ids": batch.input_ids,
                      "attention_mask": batch.attention_mask}):
        h = util.decoder_layers(model)[15].output[0]      # layer-15 residual stream
        pooled = batch.pool_response(h)                    # mean over the response span
        return pooled.norm(dim=-1).sigmoid().save()        # placeholder score — your probe goes here

scores = util.run_full_session(DATASET_NAME, detect, model=local_model,
                               remote=False, batch_size=8, max_len=256, limit=32)
print("scored", len(scores), "rows locally; first few:", scores[:5])

## 🧠 Wrapping up

- **Develop with `remote=False`** on your rented GPU; the model is loaded with `dispatch=True`.
- **Submit with `remote=True`** — NDIF runs the exact same trace code; that's the only change, and
  the contract requires it (the leaderboard has no GPU). See [`2-predicting.ipynb`](2-predicting.ipynb)
  and `submission/example.ipynb` for the submission flow, then `python submit.py --dry`.
- **Persist the HF cache** (`HF_HOME` on your mounted volume) so you don't re-download weights.
- **Terminate the Pod / instance when you're done** — idle GPUs keep billing.

*Clear all cell outputs before committing or sharing this notebook (and never commit API keys).*